<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/capstone_completed_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Refresh Opportunity Scoring

## 1. Question

**Research question:** Can a transparent baseline and a supervised model identify content pages that are at risk of a material search-impression decline in the next 30 days, using only information available at the decision date?

**Decision supported:** prioritize a review queue for content refresh candidates. This is a decision-support ranking, not a claim that refreshing a page causes recovery.

**Lane:** Refresh / Content Opportunity Scoring.

In [1]:
# Setup: run this notebook in Colab with your FlyRank Hugging Face READ token
import os, getpass, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, classification_report

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

OUT = Path("work/outputs")
OUT.mkdir(parents=True, exist_ok=True)

print("Warehouse connection ready.")


Paste your Hugging Face READ token (hf_...): ··········
Warehouse connection ready.


## 2. Data

**Release:** FlyRank internship warehouse on Hugging Face.

**Tables used:** `fact_content_daily_performance` for dated GSC performance and `dim_content` for content metadata/search volume.

**Development period:** monthly snapshots from February through May 2026. A snapshot uses only data at or before that month end. The target is the following 30-day period.

**Exclusions:** deleted/unpublished content, rows without sufficient prior history, and any fields that are future outcomes or product-decision flags. Client/content IDs are used only for grouping, joining, and reporting.

The final test snapshot is May 2026, whose June outcome is used only as the held-out label.

In [2]:
# Inspect available dates and establish the monthly snapshot windows.
dates = con.sql(f"""
SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date,
       COUNT(*) AS rows
FROM {DAILY}
""").df()
print(dates.to_string(index=False))

# Check that the development/test months have data.
month_counts = con.sql(f"""
SELECT month(report_date) AS month_num, COUNT(*) AS rows
FROM {DAILY}
WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-06-30'
GROUP BY 1 ORDER BY 1
""").df()
print("\nRows by calendar month:")
print(month_counts.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  min_date   max_date     rows
2025-01-27 2026-06-30 78835655


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows by calendar month:
 month_num     rows
         2  7355108
         3  9841378
         4 10424730
         5 11687376
         6 11694072


## 3. Methodology

### Label

For each content-month snapshot, `is_declining = 1` when impressions in the next 30 days are less than 80% of impressions in the preceding 30 days. The next-30-day window is the label only and is never supplied to the model as a feature.

### Features

All features are knowable at the snapshot date:

- previous-30-day impressions
- previous-30-day clicks
- previous-30-day average position
- previous-30-day CTR
- 90-day impressions
- 90-day clicks
- search volume
- word count
- days since content update

### Baseline

The baseline is a transparent rule: higher priority is assigned when recent impressions have already declined, with an additional staleness point for older content.

### Validation

A time-aware split is used: February–April snapshots form training data and May snapshots are the held-out test set. Both baseline and Random Forest are evaluated on exactly the same May rows.

### Leakage checks

The feature query is restricted to dates on or before each snapshot. The label query uses only the following 30 days and is kept separate from the feature matrix. No product flags or label-derived columns are used as features.

In [3]:
# Build monthly snapshots.
# Feature windows end on each snapshot date; label windows begin the following day.
snapshots = [pd.Timestamp(x) for x in ["2026-02-28","2026-03-31","2026-04-30","2026-05-31"]]

content = con.sql(f"""
SELECT client_hash_id, content_hash_id, search_volume, word_count,
       content_updated_date, is_published, is_deleted
FROM {CONTENT}
""").df()

rows = []
for snap in snapshots:
    start30 = snap - pd.Timedelta(days=29)
    start90 = snap - pd.Timedelta(days=89)
    label_start = snap + pd.Timedelta(days=1)
    label_end = snap + pd.Timedelta(days=30)

    q = f"""
    WITH base AS (
      SELECT client_hash_id, content_hash_id,
             SUM(CASE WHEN report_date BETWEEN DATE '{start30.date()}' AND DATE '{snap.date()}' THEN gsc_impressions ELSE 0 END) AS imp30,
             SUM(CASE WHEN report_date BETWEEN DATE '{start30.date()}' AND DATE '{snap.date()}' THEN gsc_clicks ELSE 0 END) AS clk30,
             AVG(CASE WHEN report_date BETWEEN DATE '{start30.date()}' AND DATE '{snap.date()}' THEN gsc_avg_position END) AS pos30,
             SUM(CASE WHEN report_date BETWEEN DATE '{start90.date()}' AND DATE '{snap.date()}' THEN gsc_impressions ELSE 0 END) AS imp90,
             SUM(CASE WHEN report_date BETWEEN DATE '{start90.date()}' AND DATE '{snap.date()}' THEN gsc_clicks ELSE 0 END) AS clk90
      FROM {DAILY}
      WHERE report_date BETWEEN DATE '{start90.date()}' AND DATE '{label_end.date()}'
      GROUP BY 1,2
    ),
    future AS (
      SELECT client_hash_id, content_hash_id,
             SUM(CASE WHEN report_date BETWEEN DATE '{label_start.date()}' AND DATE '{label_end.date()}' THEN gsc_impressions ELSE 0 END) AS future_imp30
      FROM {DAILY}
      WHERE report_date BETWEEN DATE '{label_start.date()}' AND DATE '{label_end.date()}'
      GROUP BY 1,2
    )
    SELECT b.*, f.future_imp30
    FROM base b LEFT JOIN future f USING(client_hash_id, content_hash_id)
    """
    d = con.sql(q).df()
    d["snapshot_date"] = snap
    rows.append(d)

panel = pd.concat(rows, ignore_index=True)
panel = panel.merge(content, on=["client_hash_id","content_hash_id"], how="left")

panel["ctr30"] = np.where(panel["imp30"] > 0, panel["clk30"] / panel["imp30"], np.nan)
panel["days_since_update"] = (
    panel["snapshot_date"] - pd.to_datetime(panel["content_updated_date"])
).dt.days

# Require a meaningful prior window and safe content status.
panel = panel[
    (panel["is_published"] == True) &
    (panel["is_deleted"] == False) &
    (panel["imp30"] >= 100) &
    panel["future_imp30"].notna()
].copy()

# Label is strictly future-looking and is not included in feature_cols.
panel["is_declining"] = (
    panel["future_imp30"] < 0.80 * panel["imp30"]
).astype(int)

# Baseline uses only past information.
panel["prior_momentum"] = panel["imp30"] / panel["imp90"].clip(lower=1)
panel["baseline_score"] = (
    (panel["prior_momentum"] < 0.80).astype(int) * 2
    + (panel["days_since_update"] >= 365).astype(int)
    + (panel["pos30"] >= 20).astype(int)
)
panel["baseline_action"] = np.where(panel["baseline_score"] >= 2, "REFRESH", "MONITOR")
panel["baseline_reason_code"] = np.select(
    [
        panel["prior_momentum"] < 0.80,
        panel["days_since_update"] >= 365,
        panel["pos30"] >= 20
    ],
    ["RECENT_DECLINE", "STALE_CONTENT", "LOW_VISIBILITY"],
    default="LOW_PRIORITY"
)

print("Eligible snapshot rows:", f"{len(panel):,}")
print("\nLabels by snapshot:")
print(panel.groupby("snapshot_date")["is_declining"].agg(["count","mean"]).to_string())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible snapshot rows: 396,786

Labels by snapshot:
                count      mean
snapshot_date                  
2026-02-28      77751  0.228833
2026-03-31     100817  0.497882
2026-04-30     107131  0.558615
2026-05-31     111087  0.643613


In [4]:
feature_cols = [
    "imp30","clk30","pos30","ctr30","imp90","clk90",
    "search_volume","word_count","days_since_update"
]

model_data = panel.dropna(subset=feature_cols + ["is_declining"]).copy()

train = model_data[model_data["snapshot_date"] < pd.Timestamp("2026-05-31")].copy()
test  = model_data[model_data["snapshot_date"] == pd.Timestamp("2026-05-31")].copy()

X_train, y_train = train[feature_cols], train["is_declining"]
X_test, y_test = test[feature_cols], test["is_declining"]

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

test["model_probability"] = model.predict_proba(X_test)[:,1]
test["model_action"] = np.where(test["model_probability"] >= 0.50, "REFRESH", "MONITOR")

baseline_pred = (test["baseline_action"] == "REFRESH").astype(int)
model_pred = (test["model_probability"] >= 0.50).astype(int)

results = pd.DataFrame({
    "method": ["Baseline", "Random Forest"],
    "precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, model_pred, zero_division=0)
    ],
    "recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, model_pred, zero_division=0)
    ],
    "f1": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, model_pred, zero_division=0)
    ]
})
print(results.to_string(index=False))
print("\nModel classification report:")
print(classification_report(y_test, model_pred, digits=3, zero_division=0))


       method  precision   recall      f1
     Baseline   0.653673 0.889647 0.75362
Random Forest   0.666987 0.827892 0.73878

Model classification report:
              precision    recall  f1-score   support

           0      0.555     0.342     0.423     33121
           1      0.667     0.828     0.739     52740

    accuracy                          0.640     85861
   macro avg      0.611     0.585     0.581     85861
weighted avg      0.624     0.640     0.617     85861



## 4. Results (vs baseline)

The comparison below is deliberately on the same held-out May 2026 snapshot and uses the same future decline definition for both methods.

The model is considered useful only if it improves the chosen ranking/classification measure without introducing leakage. Any result should be described as **observed performance on this held-out slice**, not as proof of causal refresh impact or Google's ranking behavior.

In [5]:
# Honest results and feature importance.
print(results.to_string(index=False))

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nFeature importance:")
print(importance.to_string(index=False))

# Leakage checks.
future_feature_overlap = [c for c in feature_cols if "future" in c.lower() or "label" in c.lower() or "declin" in c.lower()]
print("\nPotential label/future feature names:", future_feature_overlap)
print("Test snapshot:", test["snapshot_date"].min().date(), "to", test["snapshot_date"].max().date())
print("Feature data ends at snapshot date; label is future_imp30 only.")


       method  precision   recall      f1
     Baseline   0.653673 0.889647 0.75362
Random Forest   0.666987 0.827892 0.73878

Feature importance:
          feature  importance
days_since_update    0.539458
            ctr30    0.178526
            imp90    0.073952
            clk30    0.065657
       word_count    0.052724
            pos30    0.033327
            imp30    0.031764
            clk90    0.019072
    search_volume    0.005521

Potential label/future feature names: []
Test snapshot: 2026-05-31 to 2026-05-31
Feature data ends at snapshot date; label is future_imp30 only.


## 5. Limitations

- This is an observational ranking exercise; it does not establish that refreshing a page causes impressions to recover.
- Search performance histories are unbalanced across clients and content, so missing history can affect eligibility.
- The decline label is a chosen proxy for refresh risk, not a FlyRank product flag or a ground-truth business outcome.
- The held-out result comes from one time-based test period and may not generalize to every client, season, or future release.
- Search volume, position, clicks, and impressions are correlated signals; feature importance should be treated as directional rather than causal.
- The recommendations are decision-support for human review, not automatic publishing or pruning decisions.

In [6]:
# Rank the held-out content queue for the paper.
recommendations = test[[
    "client_hash_id","content_hash_id","snapshot_date",
    "model_probability","baseline_score","baseline_reason_code",
    "baseline_action","imp30","clk30","pos30","search_volume","days_since_update"
]].copy()

recommendations = recommendations.sort_values(
    ["model_probability","baseline_score"], ascending=False
).reset_index(drop=True)
recommendations["rank"] = np.arange(1, len(recommendations)+1)
recommendations["action"] = np.where(
    recommendations["model_probability"] >= 0.50, "REFRESH", "MONITOR"
)

# Public-safe output: IDs are kept for reproducibility inside the repo, not names/domains.
recommendations.to_csv(OUT / "capstone_ranked_recommendations.csv", index=False)

print(recommendations.head(20).to_string(index=False))
print(f"\nWrote {len(recommendations):,} ranked recommendations to {OUT / 'capstone_ranked_recommendations.csv'}")


         client_hash_id          content_hash_id snapshot_date  model_probability  baseline_score baseline_reason_code baseline_action   imp30  clk30     pos30  search_volume  days_since_update  rank  action
client_23a62021009f63c4 content_34550bef88b98b6e    2026-05-31           0.759767               3       RECENT_DECLINE         REFRESH  3973.0    0.0 45.012700              0                -33     1 REFRESH
client_23a62021009f63c4 content_deafa0d7f2242942    2026-05-31           0.759346               3       RECENT_DECLINE         REFRESH  6637.0    0.0 35.770591              0                -35     2 REFRESH
client_23a62021009f63c4 content_959d535a9fcc865c    2026-05-31           0.759249               3       RECENT_DECLINE         REFRESH  6527.0    0.0 43.909484              0                -34     3 REFRESH
client_23a62021009f63c4 content_cf71bae02b924aea    2026-05-31           0.759156               3       RECENT_DECLINE         REFRESH  9627.0    0.0 30.475973         

## 6. Ranked recommendations

The action playbook is:

1. **REFRESH** — prioritize human review when predicted decline risk is high.
2. **MONITOR** — keep lower-risk content in the monitoring queue.
3. Use the reason code and supporting historical signals to explain why a page entered the queue.
4. Before acting, a human should verify that the page is still relevant, technically healthy, and appropriate for refresh.

The queue is a prioritization device, not an instruction to change or remove content automatically.

In [7]:
# Paper-ready recommendation summary.
top20 = recommendations.head(20).copy()
top20["review_note"] = np.where(
    top20["baseline_reason_code"].eq("RECENT_DECLINE"),
    "Check whether the observed decline reflects a real content opportunity or a temporary search fluctuation.",
    "Check the underlying page context before taking action."
)
display(top20[[
    "rank","action","baseline_reason_code","model_probability",
    "imp30","pos30","search_volume","days_since_update","review_note"
]])


,rank,action,baseline_reason_code,model_probability,imp30,pos30,search_volume,days_since_update,review_note
0,1,REFRESH,RECENT_DECLINE,0.759767,3973.0,45.012700,0,-33,Check whether the observed decline reflects a ...
1,2,REFRESH,RECENT_DECLINE,0.759346,6637.0,35.770591,0,-35,Check whether the observed decline reflects a ...
2,3,REFRESH,RECENT_DECLINE,0.759249,6527.0,43.909484,0,-34,Check whether the observed decline reflects a ...
3,4,REFRESH,RECENT_DECLINE,0.759156,9627.0,30.475973,0,-33,Check whether the observed decline reflects a ...
4,5,REFRESH,RECENT_DECLINE,0.759062,10952.0,34.545925,0,-34,Check whether the observed decline reflects a ...
5,6,REFRESH,RECENT_DECLINE,0.758636,10178.0,34.870589,0,-34,Check whether the observed decline reflects a ...
6,7,REFRESH,RECENT_DECLINE,0.757914,2576.0,39.856694,0,-34,Check whether the observed decline reflects a ...
7,8,REFRESH,RECENT_DECLINE,0.757610,16098.0,44.494453,0,-33,Check whether the observed decline reflects a ...
8,9,REFRESH,RECENT_DECLINE,0.757457,2777.0,33.793757,0,-18,Check whether the observed decline reflects a ...
9,10,REFRESH,RECENT_DECLINE,0.757393,12104.0,28.963457,0,-26,Check whether the observed decline reflects a ...


## 7. Artifacts the paper embeds

The notebook produces a held-out model-vs-baseline table, feature-importance table, and ranked recommendation CSV. These are the reproducibility artifacts for the deployed paper.

In [8]:
# Save paper artifacts and a compact metrics receipt.
metrics = {
    "test_snapshot": "2026-05-31",
    "train_snapshots": ["2026-02-28","2026-03-31","2026-04-30"],
    "label": "future_30d_impressions < 0.80 * prior_30d_impressions",
    "test_rows": int(len(test)),
    "decline_rate_test": float(y_test.mean()),
    "baseline_precision": float(results.loc[results.method=="Baseline","precision"].iloc[0]),
    "baseline_recall": float(results.loc[results.method=="Baseline","recall"].iloc[0]),
    "baseline_f1": float(results.loc[results.method=="Baseline","f1"].iloc[0]),
    "model_precision": float(results.loc[results.method=="Random Forest","precision"].iloc[0]),
    "model_recall": float(results.loc[results.method=="Random Forest","recall"].iloc[0]),
    "model_f1": float(results.loc[results.method=="Random Forest","f1"].iloc[0]),
    "feature_count": len(feature_cols),
    "leakage_feature_names_found": future_feature_overlap
}
with open(OUT / "capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

results.to_csv(OUT / "capstone_model_vs_baseline.csv", index=False)
importance.to_csv(OUT / "capstone_feature_importance.csv", index=False)

print(json.dumps(metrics, indent=2))


{
  "test_snapshot": "2026-05-31",
  "train_snapshots": [
    "2026-02-28",
    "2026-03-31",
    "2026-04-30"
  ],
  "label": "future_30d_impressions < 0.80 * prior_30d_impressions",
  "test_rows": 85861,
  "decline_rate_test": 0.6142486111272871,
  "baseline_precision": 0.653673079870157,
  "baseline_recall": 0.8896473265073948,
  "baseline_f1": 0.7536199294886724,
  "model_precision": 0.6669874585643799,
  "model_recall": 0.8278915434205537,
  "model_f1": 0.7387798956033265,
  "feature_count": 9,
  "leakage_feature_names_found": []
}


## ML-12 closing material

### 5-minute demo outline
1. State the refresh-risk question and decision.
2. Explain the monthly snapshot and future-30-day decline label.
3. Show the transparent baseline and why the model is compared against it.
4. Show held-out baseline vs Random Forest results.
5. Show the ranked queue and one or two examples.
6. Close with limitations: observed, directional, decision-support—not causal proof.

### Social-post cut
Built a search-intelligence capstone on real FlyRank data: a time-aware model and transparent baseline rank content pages for refresh review. The evaluation uses a held-out future window and explicit leakage checks. Results are framed as observed decision-support evidence, not causal claims about search algorithms or refresh impact.

### Employer-facing summary
Built an end-to-end search-intelligence workflow from warehouse data to a transparent baseline, time-aware Random Forest model, held-out evaluation, and ranked content recommendations. I treated leakage and unbalanced history as first-class validation concerns and compared the model with a simple rule on the same test period. The final output is a reproducible decision-support system rather than a claim of causal impact.

## Self-check

- [ ] Every section above is filled — markdown thinking AND code that backs it.
- [ ] The notebook runs top to bottom with no errors in Colab.
- [ ] No client names, domains, private queries, credentials, or raw exports are included.
- [ ] Features are available at the decision date; future labels are kept separate.
- [ ] Baseline and model use the same held-out test snapshot.
- [ ] Claims use observed / directional / decision-support language.
- [ ] Ranked recommendations and metrics artifacts are generated.
- [ ] The notebook is committed under `work/notebooks/capstone.ipynb`.
- [ ] The deployed paper contains all required sections, including Abstract and Acknowledgments & data credit.
- [ ] `submission/paper_url.txt` contains exactly the deployed paper URL.